# Asuinkiinteistöjen energiankysyntäjärjestelmän estimointi näennäisesti riippumattomalla regressiolla

## Tiivistelmä

Alueellinen sähkö- ja kaasulaitos estimoi kahden yhtälön asuinkiinteistöjen **energiankysyntäjärjestelmän** — sähkön ja maakaasun budjettiosuudet oman hinnan, ristihinnan ja kotitalouden tulojen funktioina — käyttäen `PROC SYSLIN` -proseduuria näennäisesti riippumattoman regression (SUR) menetelmällä. Kahdella osuusyhtälöllä on yhteinen kotitalouden budjetti, joten niiden virhetermit ovat ristikorreloituneita; SUR estimoi yhtälöt yhdessä hyödyntääkseen tuota korrelaatiota, palauttaa merkiltään johdonmukaiset oman ja ristihinnan vaikutukset sekä tuottaa yhtälöiden väliset kovarianssi- ja rajoitustestit, joita kysyntäanalyytikko tarvitsee.

## Tietolähteet

| Aineisto | Rivejä | Rakeisuus | Keskeiset muuttujat | Kuvaus |
|---------|------|-------|---------------|-------------|
| `energy` | 60 | yksi rivi kutakin kuukausittaista markkinahavaintoa kohti | `month`, `lp_elec`, `lp_gas`, `lincome`, `w_elec`, `w_gas` | Synteettinen kuukausittainen asuinkiinteistöjen energiamarkkinapaneeli. `lp_elec`/`lp_gas` ovat sähkön ja maakaasun logaritmisia reaalihintoja; `lincome` on kotitalouden logaritminen reaalitulo; `w_elec`/`w_gas` ovat sähkön ja maakaasun menobudjettiosuudet, jotka on tuotettu tunnetusta AIDS-tyyppisestä kysyntärakenteesta lisättynä korreloitunut yhtälöiden välinen kohina. |

# Asuinkiinteistöjen energiankysyntäjärjestelmän estimointi näennäisesti riippumattomalla regressiolla

Alueellinen kaasu- ja sähkölaitos haluaa ymmärtää, miten kotitaloudet jakavat energiabudjettinsa uudelleen **sähkön** ja **maakaasun** välillä suhteellisten hintojen ja tulojen muuttuessa. Luonteva viitekehys on **kysyntäjärjestelmä**: joukko budjettiosuusyhtälöitä, jotka estimoidaan yhdessä.

Kaksi piirrettä tekee yhteisestä estimoinnista oikean työkalun:

- Osuusyhtälöt ammentavat yhteisestä kotitalousbudjetista, joten niiden virhetermit ovat **ristikorreloituneita** — kun kotitalous käyttää enemmän sähköön, se käyttää vähemmän kaasuun. Yhtälöiden estimointi yhdessä **näennäisesti riippumattomalla regressiolla (SUR)** hyödyntää tuota korrelaatiota tehokkuuden vuoksi.
- Taloustieteellinen teoria asettaa **lineaarisia rajoituksia** (yhteenlaskettavuus, homogeenisuus, symmetria), joita järjestelmäestimaattori voi valvoa suoraan.

`PROC SYSLIN` `SUR`-optiolla on rakennettu juuri tähän tilanteeseen. Se sovittaa kunkin osuusyhtälön, estimoi yhtälöiden välisen virhekovarianssin ja sovittaa järjestelmän uudelleen tuolla kovarianssilla tuottaakseen tehokkaat, teorian kanssa johdonmukaiset joustot — yhdessä mallien välisen kovarianssimatriisin ja yhteisten rajoitustestien kanssa.

Tässä muistikirjassa me:

1. Tuotamme synteettisen kuukausittaisen energiamarkkinapaneelin tunnetusta osuusrakenteesta.
2. Sovitamme kahden yhtälön osuusjärjestelmän SUR-menetelmällä.
3. Vertaamme yhteistä SUR-sovitusta yhtälö kerrallaan tehtyyn OLS-sovitukseen.
4. Asetamme homogeenisuusrajoituksen ja luemme sen yhteisen F-testin.
5. Poimimme kerroin­taulukon joustoraportointia varten.

## Vaihe 1 — Tuota synteettinen energiamarkkinapaneeli

Simuloimme 60 kuukausittaista havaintoa pienestä **kahden hyödykkeen energiankysyntäjärjestelmästä** (sähkö ja maakaasu). Tietoja tuottava prosessi on linearisoitu, AIDS-tyyppinen budjettiosuusmalli:

```
w_elec = a1 + g11*lp_elec + g12*lp_gas + b1*lincome + e1
w_gas  = a2 + g21*lp_elec + g22*lp_gas + b2*lincome + e2
```

Rakennamme tarkoituksella sisään **yhtälöiden välisen korrelaation**: kun kotitaloudet käyttävät enemmän sähköön, ne käyttävät vähemmän kaasuun, joten `e1` ja `e2` ovat negatiivisesti korreloituneita. Yhteinen energiamarkkinoiden hintatekijä ajaa myös molempia logaritmisia hintoja yhdessä. Nämä ovat piirteitä, jotka palkitsevat yhteisen SUR-estimoinnin verrattuna kunkin yhtälön sovittamiseen yksinään.

In [1]:
TIEDOT energy;
   CALL streaminit(70123);

   /* True structural coefficients (linearized AIDS share system) */
   a1  = 0.55;  g11 =  0.12; g12 = -0.08; b1 = -0.030;
   a2  = 0.45;  g21 = -0.08; g22 =  0.10; b2 = -0.025;

   TEE month = 1 ASTI 60;
      /* A common energy-market price factor drives BOTH prices,
         creating the collinearity that makes the problem ill-posed. */
      energy_factor = rand('NORMAL', 0, 0.15);

      lp_elec = 4.6 + energy_factor + rand('NORMAL', 0, 0.05);
      lp_gas  = 4.2 + energy_factor + rand('NORMAL', 0, 0.05);
      lincome = 10.8 + 0.004*month + rand('NORMAL', 0, 0.06);

      /* Negatively correlated cross-equation errors (budget tradeoff) */
      shock = rand('NORMAL', 0, 0.010);
      e1 =  shock + rand('NORMAL', 0, 0.006);
      e2 = -shock + rand('NORMAL', 0, 0.006);

      w_elec = a1 + g11*lp_elec + g12*lp_gas + b1*lincome + e1;
      w_gas  = a2 + g21*lp_elec + g22*lp_gas + b2*lincome + e2;

      TULOSTE;
   LOPPU;

   SÄILYTÄ month lp_elec lp_gas lincome w_elec w_gas;
SUORITA;

PROSEDUURI KESKIARVOT TIEDOT=energy n mean std MIN MAX maxdec=3;
   MUUTTUJA w_elec w_gas lp_elec lp_gas lincome;
SUORITA;

                                                  The MEANS Procedure

 Variable        N           Mean     Std Dev     Minimum     Maximum
 --------------------------------------------------------------------
 W_ELEC         60          0.440       0.011       0.417       0.464
 W_GAS          60          0.228       0.014       0.198       0.256
 LP_ELEC        60          4.617       0.142       4.354       4.911
 LP_GAS         60          4.211       0.162       3.818       4.566
 LINCOME        60         10.916       0.088      10.723      11.174
 --------------------------------------------------------------------



NOTE: DATA energy


NOTE: Wrote energy (60 rows, 6 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Vaihe 2 — Kysyntäjärjestelmän perustason SUR-estimointi

Estimoimme molemmat osuusyhtälöt yhdessä. `SUR`-optio kertoo `PROC SYSLIN` -proseduurille, että sen tulee estimoida yhtälöiden välinen virhekovarianssi ja käyttää sitä tehokkaaseen toteutettuun GLS-sovitukseen. Kaksi nimettyä `MODEL`-lausetta (`elec` ja `gas`) määrittävät järjestelmän; kumpikin regressoi budjettiosuuden kahdella logaritmisella hinnalla ja logaritmisella tulolla.

SYSLIN raportoi kunkin yhtälön parametriestimaatit ja lopuksi **mallien välisen kovarianssimatriisin (Cross-Model Covariance Matrix)** — kahden yhtälön virhetermien estimoidun kovarianssin, jota SUR hyödyntää.

In [2]:
PROSEDUURI syslin TIEDOT=energy ON;
   elec: MODEL w_elec = lp_elec lp_gas lincome;
   gas:  MODEL w_gas  = lp_elec lp_gas lincome;
SUORITA;


                       The SYSLIN Procedure

                  SUR Estimation

  Model elec Dependent Variable: W_ELEC

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.150270       5.33      0.0000
  LP_ELEC        1      0.112463    0.021244       5.29      0.0000
  LP_GAS         1     -0.097938    0.018646      -5.25      0.0000
  LINCOME        1     -0.042842    0.013238      -3.24      0.0020

  Model gas Dependent Variable: W_GAS

  Number of Observations                       60
  SSE                                      0.

NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (SUR)
NOTE: PROC SYSLIN completed.


## Vaihe 3 — Vertaa yhtälö kerrallaan tehtyyn OLS-sovitukseen

Nähdäksemme, mitä SUR meille tuo, sovitamme samat kaksi yhtälöä uudelleen pienimmän neliösumman menetelmällä (oletusmenetelmä, yksi yhtälö kerrallaan). OLS jättää huomiotta yhtälöiden välisen virhekorrelaation, joten se on tarkentuva mutta vähemmän tehokas kuin SUR, kun tuo korrelaatio on nollasta poikkeava — kuten se tässä rakenteeltaan on.

Kahden kerrointaulukon vertailu osoittaa, miten estimaatit ja niiden keskivirheet muuttuvat, kun järjestelmän rakenne otetaan huomioon.

In [3]:
PROSEDUURI syslin TIEDOT=energy ols;
   elec: MODEL w_elec = lp_elec lp_gas lincome;
   gas:  MODEL w_gas  = lp_elec lp_gas lincome;
SUORITA;


                       The SYSLIN Procedure

                  OLS Estimation

  Model elec Dependent Variable: W_ELEC

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.155544       5.15      0.0000
  LP_ELEC        1      0.112463    0.021989       5.11      0.0000
  LP_GAS         1     -0.097938    0.019300      -5.07      0.0000
  LINCOME        1     -0.042842    0.013702      -3.13      0.0028

  Model gas Dependent Variable: W_GAS

  Number of Observations                       60
  SSE                                      0.

NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (OLS)
NOTE: PROC SYSLIN completed.


## Vaihe 4 — Aseta taloustieteellinen teoria ja testaa se

Kysyntäteoria sanoo, että nimellisten hinta- ja tulovaikutusten tulisi noudattaa **nolla-asteista homogeenisuutta**: kaikkien hintojen ja tulojen skaalaaminen jättää reaalikysynnän muuttumattomaksi, joten yhtälön sisäisten hinta- ja tulokerrointen summa on nolla. Asetamme tuon sähkön osuudelle `RESTRICT`-lauseella.

SYSLIN sovittaa järjestelmän uudelleen rajoituksen alaisena ja raportoi **rajoitustulosten (Restriction Results)** F-testin siitä, onko rajoitus yhteensopiva aineiston kanssa — suoran, yhteisen testin homogeenisuushypoteesille.

In [4]:
PROSEDUURI syslin TIEDOT=energy ON;
   elec: MODEL w_elec = lp_elec lp_gas lincome;
   gas:  MODEL w_gas  = lp_elec lp_gas lincome;

   /* Homogeneity on the electricity share equation:
      price and income coefficients sum to zero. */
   restrict lp_elec + lp_gas + lincome = 0;
SUORITA;


                       The SYSLIN Procedure

                  SUR Estimation

  Model elec Dependent Variable: W_ELEC

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.150270       5.33      0.0000
  LP_ELEC        1      0.112463    0.021244       5.29      0.0000
  LP_GAS         1     -0.097938    0.018646      -5.25      0.0000
  LINCOME        1     -0.042842    0.013238      -3.24      0.0020

  Model gas Dependent Variable: W_GAS

  Number of Observations                       60
  SSE                                      0.

NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (SUR)
NOTE: PROC SYSLIN completed.


## Vaihe 5 — Tallenna estimaatit joustoraportointia varten

Loppuraporttia varten tallennamme suppenneet kertoimet `OUTEST=`-optiolla. Tuloksena syntyvä aineisto sisältää yhden rivin yhtälöä kohti vakiotermin ja kulmakerroin­estimaatit sekä sovitustunnusluvut, jotka syöttävät laitoksen omat ja ristihintajoustolaskelmat otoskeskiarvo-osuuksilla. Tulostamme sen dokumentointia varten.

In [5]:
PROSEDUURI syslin TIEDOT=energy ON outest=demand_est;
   elec: MODEL w_elec = lp_elec lp_gas lincome;
   gas:  MODEL w_gas  = lp_elec lp_gas lincome;
SUORITA;

PROSEDUURI TULOSTA TIEDOT=demand_est noobs;
   OTSIKKO "SUR demand-system coefficient estimates";
SUORITA;
OTSIKKO;


                       The SYSLIN Procedure

                  SUR Estimation

  Model elec Dependent Variable: W_ELEC

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.150270       5.33      0.0000
  LP_ELEC        1      0.112463    0.021244       5.29      0.0000
  LP_GAS         1     -0.097938    0.018646      -5.25      0.0000
  LINCOME        1     -0.042842    0.013238      -3.24      0.0020

  Model gas Dependent Variable: W_GAS

  Number of Observations                       60
  SSE                                      0.

NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (SUR)
NOTE: Wrote OUTEST= dataset demand_est (2 rows).
NOTE: PROC SYSLIN completed.
NOTE: PROC PRINT data=demand_est

NOTE: PROC PRINT completed: 2 observations printed, 12 variables


## Tulosten tulkinta

**Merkkijohdonmukaisuus.** SUR-estimaatit palauttavat aineistoon rakennetun substituutiorakenteen. Sähkön osuuden yhtälössä **oman logaritmisen hinnan kerroin on positiivinen** (`LP_ELEC` = 0.112), kun taas **ristihinnan kerroin on negatiivinen** (`LP_GAS` = -0.098); kaasun osuuden yhtälössä kuvio peilaa tätä (oma `LP_GAS` = 0.114, risti `LP_ELEC` = -0.075). Jokainen oman ja ristihinnan vaikutus on tilastollisesti merkitsevä (kukin `Pr > |t|` alle 0.005), joten substituutiomerkit ovat vahvasti tunnistettuja eivätkä kohinaisen sovituksen tuottamia harha-artefakteja.

**SUR hyödyntää yhtälöiden välistä korrelaatiota.** **Mallien välinen kovarianssimatriisi** osoittaa negatiivisen diagonaalin ulkopuolisen alkion (-0.000068): kotitaloudet vaihtavat sähkömenot kaasumenoja vastaan, juuri kuten on rakennettu. Koska tuo kovarianssi on nollasta poikkeava, yhteinen SUR-estimointi on tehokkaampaa kuin kunkin yhtälön sovittaminen yksinään — vaiheen 2 SUR-keskivirheet ovat kauttaaltaan hieman pienempiä kuin vaiheen 3 yhtälö kerrallaan tehdyt OLS-keskivirheet (esimerkiksi kaasun `LP_ELEC`-keskivirhe laskee OLS:n 0.0264:stä SUR:n 0.0255:een), samalla kun piste-estimaatit pysyvät muuttumattomina. Tuo tiukempi päättely on järjestelmän mallintamisen tuotto verrattuna kahteen erilliseen regressioon.

**Teoria pitää paikkansa.** Sähkön osuudelle asetettu **nolla-asteinen homogeenisuus** (hinta- ja tulokerrointen summa nolla) `RESTRICT`-lauseella tuotti rajoitustulosten F-testin 2.14 arvolla `Pr > F` = 0.149. Rajoitusta **ei hylätä**, joten kuluttajien kysyntäteorian homogeenisuusennuste on yhteensopiva tämän otoksen kanssa — laitos voi viedä rajoitetut, teorian kanssa johdonmukaiset estimaatit raportointiin luottavaisin mielin.

**Operatiivinen käyttö.** `OUTEST=`-taulukko tallentaa yhden rivin yhtälöä kohti vakiotermin ja kulmakerroin­estimaatit sekä sovitustunnusluvut. Nuo kertoimet muuntuvat suoraan oman ja ristihinnan joustoiksi sekä tulo- (meno-) joustoksi otoskeskiarvo-osuuksilla (`W_ELEC` ~ 0.44, `W_GAS` ~ 0.23 vaiheesta 1), antaen laitokselle puolustettavan, teorian kanssa johdonmukaisen perustan kysyntäennustamiselle ja hinnoittelun suunnitteluskenaarioille.